In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    accuracy_score
)
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# 1. ЗАГРУЗКА И ПРЕДВАРИТЕЛЬНЫЙ АНАЛИЗ ДАННЫХ
# ============================================================================
print("[1/4] Загрузка и предварительный анализ данных...")

try:
    df = pd.read_csv('cardio_train.csv', sep=';')
    print(f"✓ Загружено {df.shape[0]} записей и {df.shape[1]} признаков.")
except FileNotFoundError:
    raise FileNotFoundError("Файл 'cardio_train.csv' не найден. Скачайте его с Kaggle.")

# Преобразуем возраст из дней в полные годы
df['age_years'] = (df['age'] // 365.25).astype(int)

df = df.drop(columns=['id', 'age'])

# ============================================================================
# 2. ОЧИСТКА ДАННЫХ И ИНЖЕНЕРИЯ ПРИЗНАКОВ
# ============================================================================
print("[2/4] Очистка данных и создание новых признаков...")

# Фильтрация выбросов по медицински обоснованным критериям
initial_count = len(df)
df = df[
    (df['height'] >= 120) & (df['height'] <= 220) &      # рост: 120–220 см
    (df['weight'] >= 30) & (df['weight'] <= 200) &       # вес: 30–200 кг
    (df['ap_hi'] >= 60) & (df['ap_hi'] <= 220) &         # верхнее давление
    (df['ap_lo'] >= 40) & (df['ap_lo'] <= 150) &         # нижнее давление
    (df['ap_hi'] > df['ap_lo'])                          # верхнее > нижнего
]
cleaned_count = len(df)
print(f"✓ Удалено {initial_count - cleaned_count} выбросов.")

# Расчёт индекса массы тела (ИМТ)
df['bmi'] = df['weight'] / ((df['height'] / 100) ** 2)

# Удаляем исходные признаки роста и веса (ИМТ информативнее)
df = df.drop(columns=['height', 'weight'])

# Разделяем признаки и целевую переменную
X = df.drop(columns=['cardio'])
y = df['cardio']

# Вывод баланса классов
print(f"✓ Распределение целевой переменной: {y.value_counts().to_dict()}")

# ============================================================================
# 3. НАСТРОЙКА МОДЕЛЕЙ И КРОСС-ВАЛИДАЦИИ
# ============================================================================
print("[3/4] Настройка моделей и схемы валидации...")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'LogisticRegression': Pipeline([
        ('scaler', StandardScaler()), 
        ('clf', LogisticRegression(
            random_state=42,
            max_iter=1000,
            solver='liblinear'  
        ))
    ]),
    'RandomForest': RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight='balanced',  
        n_jobs=-1              
    )
}

# ============================================================================
# 4. ОБУЧЕНИЕ, ОЦЕНКА И ВЫВОД РЕЗУЛЬТАТОВ
# ============================================================================
print("[4/4] Обучение моделей и оценка качества...\n")

results = {}

for name, model in models.items():
    print(f"=== Оценка модели: {name} ===")
    scores = {
        'roc_auc': [], 'f1': [], 'precision': [],
        'recall': [], 'accuracy': []
    }
    
    for fold, (train_idx, val_idx) in enumerate(cv.split(X, y), 1):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model.fit(X_train, y_train)
        
        y_pred = model.predict(X_val)
        y_proba = model.predict_proba(X_val)[:, 1]

        scores['roc_auc'].append(roc_auc_score(y_val, y_proba))
        scores['f1'].append(f1_score(y_val, y_pred))
        scores['precision'].append(precision_score(y_val, y_pred))
        scores['recall'].append(recall_score(y_val, y_pred))
        scores['accuracy'].append(accuracy_score(y_val, y_pred))

    avg_scores = {metric: np.mean(vals) for metric, vals in scores.items()}
    results[name] = avg_scores

    print(f"  ROC-AUC:    {avg_scores['roc_auc']:.4f}")
    print(f"  F1-score:   {avg_scores['f1']:.4f}")
    print(f"  Precision:  {avg_scores['precision']:.4f}")
    print(f"  Recall:     {avg_scores['recall']:.4f}")
    print(f"  Accuracy:   {avg_scores['accuracy']:.4f}\n")

# ============================================================================
# 5. ФИНАЛЬНЫЙ ОТЧЁТ: СРАВНЕНИЕ МОДЕЛЕЙ
# ============================================================================
print("=== ИТОГОВОЕ СРАВНЕНИЕ МОДЕЛЕЙ ===")
best_model_name = max(results.keys(), key=lambda k: results[k]['roc_auc'])
print(f"🏆 Лучшая модель по ROC-AUC: {best_model_name}")

print("\nМетрики моделей:")
for name, scores in results.items():
    print(f"\n{name}:")
    for metric, value in scores.items():
        print(f"  {metric.upper()}: {value:.4f}")

print(f"\n=== Генерация полного классификационного отчёта для {best_model_name} ===")
best_model = models[best_model_name]
best_model.fit(X, y)
y_full_pred = best_model.predict(X)
print(classification_report(y, y_full_pred, target_names=['Нет ССЗ', 'Есть ССЗ']))

[1/4] Загрузка и предварительный анализ данных...
✓ Загружено 70000 записей и 13 признаков.
[2/4] Очистка данных и создание новых признаков...
✓ Удалено 1397 выбросов.
✓ Распределение целевой переменной: {0: 34669, 1: 33934}
[3/4] Настройка моделей и схемы валидации...
[4/4] Обучение моделей и оценка качества...

=== Оценка модели: LogisticRegression ===
  ROC-AUC:    0.7911
  F1-score:   0.7076
  Precision:  0.7539
  Recall:     0.6667
  Accuracy:   0.7275

=== Оценка модели: RandomForest ===
  ROC-AUC:    0.7454
  F1-score:   0.6862
  Precision:  0.6826
  Recall:     0.6898
  Accuracy:   0.6879

=== ИТОГОВОЕ СРАВНЕНИЕ МОДЕЛЕЙ ===
🏆 Лучшая модель по ROC-AUC: LogisticRegression

Метрики моделей:

LogisticRegression:
  ROC_AUC: 0.7911
  F1: 0.7076
  PRECISION: 0.7539
  RECALL: 0.6667
  ACCURACY: 0.7275

RandomForest:
  ROC_AUC: 0.7454
  F1: 0.6862
  PRECISION: 0.6826
  RECALL: 0.6898
  ACCURACY: 0.6879

=== Генерация полного классификационного отчёта для LogisticRegression ===
         